<a href="https://colab.research.google.com/github/eshwarjyEnder-Cyber/Ender/blob/main/Classical_model_for_Scallable_ML_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_openml

In [34]:
adult = fetch_openml(
    'adult',
    version = 2,
    as_frame = True,
    parser = 'auto'
)
raw_df = adult.frame.copy()

In [35]:
raw_df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K


In [36]:
print("Dataset load successfully")
print('Rows:', len(raw_df))
print("Columns:", len(raw_df.columns))
raw_df.info()

Dataset load successfully
Rows: 48842
Columns: 15
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   age             48842 non-null  int64   
 1   workclass       46043 non-null  category
 2   fnlwgt          48842 non-null  int64   
 3   education       48842 non-null  category
 4   education-num   48842 non-null  int64   
 5   marital-status  48842 non-null  category
 6   occupation      46033 non-null  category
 7   relationship    48842 non-null  category
 8   race            48842 non-null  category
 9   sex             48842 non-null  category
 10  capital-gain    48842 non-null  int64   
 11  capital-loss    48842 non-null  int64   
 12  hours-per-week  48842 non-null  int64   
 13  native-country  47985 non-null  category
 14  class           48842 non-null  category
dtypes: category(9), int64(6)
memory usage: 2.7 MB


In [37]:
Expected_Columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "class"
]
Target = 'class'

In [38]:
if raw_df.empty:
  raise ValueError('Pipeline stopped. No data received')

print('✓ Data availability check passed')

✓ Data availability check passed


In [39]:
# Data cleaning
missing_columns = [
    column
    for column in Expected_Columns
    if column not in raw_df.columns
]
if missing_columns:
  raise ValueError( f"pip0eline stopped. Missing Columns: {missing_columns}")

print('✓ Expected column checked')

✓ Expected column checked


In [40]:
# Fill the missing values in data
df = raw_df.replace('?', pd.NA)
missing_values = df.isna().sum()

# Missing values treatment is checked and implemented on all columns
missing_values[missing_values > 0]


,0
workclass,2799
occupation,2809
native-country,857


In [41]:
if df[Target].nunique() < 2 or df[Target].nunique() > 2:
  raise ValueError('Pipeline stopped: target has fewer or more than two classes.')

print(' ✓  Target validation is passed')


 ✓  Target validation is passed


In [42]:
prepared_df = (
    df
    .drop_duplicates()
    .dropna() # Can be replaced with some missing value strategy
    .reset_index(drop= True)
)
print("✓  Prepared dataset shape:", prepared_df.shape)

✓  Prepared dataset shape: (45175, 15)


In [43]:
# output Validation

assert not prepared_df.empty
assert Target in prepared_df.columns
assert prepared_df.isna().sum().sum() == 0
assert prepared_df[Target].nunique() == 2

print("✓  prepared dataset passed output validation")

✓  prepared dataset passed output validation


In [44]:
X = prepared_df.drop(columns = [Target])

Y = (
    prepared_df[Target]
    .astype(str)
    .str.strip()
    .eq(">50K")
    .astype(int)
)

print("Input shape:", X.shape)
print("Target shape:", Y.shape)


Input shape: (45175, 14)
Target shape: (45175,)


In [45]:
numeric_features = X.select_dtypes(
    include = ["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude = ['number']
).columns.tolist()

print('Numeric featires:')
print(numeric_features)

print("\n Categorical features:")
print(categorical_features)

Numeric featires:
['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

 Categorical features:
['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']


In [46]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers = [
        (
            "numeric",
            "passthrough",
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown = 'ignore',
                sparse_output = False
            ),
            categorical_features
        )
    ]
)

In [47]:
preprocessor

ColumnTransformer(transformers=[('numeric', 'passthrough',
                                 ['age', 'fnlwgt', 'education-num',
                                  'capital-gain', 'capital-loss',
                                  'hours-per-week']),
                                ('categorical',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['workclass', 'education', 'marital-status',
                                  'occupation', 'relationship', 'race', 'sex',
                                  'native-country'])])

In [48]:
# Train the model

from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size = 0.25,
    random_state = 42,
    stratify = Y
)

print("Training rows:", len(X_train_raw))
print("Evaluation rows:", len(X_test_raw))


Training rows: 33881
Evaluation rows: 11294


In [49]:
X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

#missing value - fit         + transform
 #Train          mean/median.   fillna()
 #Test                          transform
                                fillna()
                                

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)
X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

In [50]:
feature_names = preprocessor.get_feature_names_out() # Column names

X_train = pd.DataFrame(
    X_train,
    columns = feature_names
)
X_test = pd.DataFrame(
    X_test,
    columns = feature_names
)

print("Model-ready training shape:", X_train.shape)
print("Model-ready evaluation shape", X_test.shape)

Model-ready training shape: (33881, 104)
Model-ready evaluation shape (11294, 104)


In [51]:
assert X_train.shape[1] == X_test.shape[1]
assert list(X_train.columns) == list(X_test.columns)

assert not X_train.isna().any().any()
assert not X_test.isna().any().any()

assert len(X_train) == len(Y_train)
assert len(X_test) == len(Y_test)

print("✓ Feature validation passed")

✓ Feature validation passed


In [52]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter = 500,
    random_state = 42
)

model.fit(X_train, Y_train)

print("✓ Model training completed")

✓ Model training completed


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [56]:
print("Model type:", type(model).__name__)
print("Number of model inputs;", model.n_features_in_)

Model type: LogisticRegression
Number of model inputs; 104


In [57]:
assert model is not None
assert model.n_features_in_ == X_train.shape[1]

print("✓ Model artifact validation passed")

✓ Model artifact validation passed


Python Scripts:
- Main File - Where you need to trigger
- Data_loading.py - load data and data validation
- preprocess.py - load validated data, preprocess and validated
- modelling.py - preprocessed data , train-test, modelling
- evaluation.py - model, evaluate model and validate the model